In [ ]:
# ============================================================
# Configuration
# ============================================================
# All asset paths and spatial parameters used across the
# notebook are defined here. The VERSION suffix is appended
# to every output asset name.
# ============================================================

VERSION = "_10m"

# Project
PROJECT_ID = "bar-project-457815"
ASSET_ROOT = f"projects/{PROJECT_ID}/assets"

# Spatial parameters
SCALE      = 10
CRS        = "EPSG:32636"
MAX_PIXELS = 1e13

# Cloud masking and temporal coverage
CLEAR_THRESH = 0.60
DATE_START   = "2017-01-01"
DATE_END     = "2025-12-31"

# Output asset paths
ASSET_COLLECTION       = f"{ASSET_ROOT}/s2_monthly_image_collection{VERSION}"
LABEL_COLLECTION       = f"{ASSET_ROOT}/fire_monthly_labels{VERSION}"
BASELINE_MEAN_FOLDER   = f"{ASSET_ROOT}/seasonal_baseline_mean{VERSION}"
BASELINE_MEDIAN_FOLDER = f"{ASSET_ROOT}/seasonal_baseline_median{VERSION}"
BASELINE_STD_FOLDER    = f"{ASSET_ROOT}/seasonal_baseline_std{VERSION}"
ANOMALY_FOLDER         = f"{ASSET_ROOT}/monthly_anomaly{VERSION}"

# Source assets (region of interest, masks, fire ground truth)
ROI_FC_ID         = f"{ASSET_ROOT}/gaza_envelope_for_decell_Dissolve"
LANDUSE_FC_ID     = f"{ASSET_ROOT}/otef_landuse"
HUMAN_INFRA_FC_ID = f"{ASSET_ROOT}/human_infrastructure_mask_utm36n"
FIRES_TRUTH_FC_ID = f"{ASSET_ROOT}/fires_scars_otef_6991"

print("=" * 60)
print(f"VERSION    = '{VERSION}'")
print(f"Resolution = {SCALE} m")
print("=" * 60)
print(f"S2 composites    -> {ASSET_COLLECTION}")
print(f"Fire labels      -> {LABEL_COLLECTION}")
print(f"Baseline mean    -> {BASELINE_MEAN_FOLDER}")
print(f"Baseline median  -> {BASELINE_MEDIAN_FOLDER}")
print(f"Baseline std     -> {BASELINE_STD_FOLDER}")
print(f"Anomaly          -> {ANOMALY_FOLDER}")
print("=" * 60)

VERSION    = '_10m'
Resolution = 10 m
S2 composites    -> projects/bar-project-457815/assets/s2_monthly_image_collection_10m
Fire labels      -> projects/bar-project-457815/assets/fire_monthly_labels_10m
Baseline mean    -> projects/bar-project-457815/assets/seasonal_baseline_mean_10m
Baseline median  -> projects/bar-project-457815/assets/seasonal_baseline_median_10m
Baseline std     -> projects/bar-project-457815/assets/seasonal_baseline_std_10m
Anomaly          -> projects/bar-project-457815/assets/monthly_anomaly_10m


In [ ]:
# ============================================================
# Install dependencies and initialize Earth Engine
# ============================================================

!pip install -q -U geemap

import ee
import geemap

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

print("Earth Engine initialized for project:", PROJECT_ID)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 53.7 MB/s eta 0:00:00
Earth Engine initialized for project: bar-project-457815


In [ ]:
# ============================================================
# Delete the original 20 m assets from the Earth Engine project.
# Uses the Python API, which does not prompt for confirmation
# on each deletion.
# ============================================================
'''
ASSET_ROOT = "projects/bar-project-457815/assets"

collections_to_delete = [
    f"{ASSET_ROOT}/s2_monthly_image_collection",
    f"{ASSET_ROOT}/fire_monthly_labels",
    f"{ASSET_ROOT}/seasonal_baseline_mean",
    f"{ASSET_ROOT}/seasonal_baseline_median",
    f"{ASSET_ROOT}/seasonal_baseline_std",
    f"{ASSET_ROOT}/monthly_anomaly",
]

for path in collections_to_delete:
    print(f"Processing: {path}")

    try:
        # List all images inside the collection
        result = ee.data.listAssets({"parent": path})
        images = result.get("assets", [])

        print(f"  Found {len(images)} images")

        # Delete each image without confirmation
        for img in images:
            ee.data.deleteAsset(img["name"])

        # Delete the now-empty collection itself
        ee.data.deleteAsset(path)
        print(f"  Deleted collection: {path}\n")

    except Exception as e:
        print(f"  Error: {e}\n")

print("Cleanup complete.")
'''

Processing: projects/bar-project-457815/assets/s2_monthly_image_collection
  Found 88 images
  Deleted collection: projects/bar-project-457815/assets/s2_monthly_image_collection

Processing: projects/bar-project-457815/assets/fire_monthly_labels
  Found 108 images
  Deleted collection: projects/bar-project-457815/assets/fire_monthly_labels

Processing: projects/bar-project-457815/assets/seasonal_baseline_mean
  Found 12 images
  Deleted collection: projects/bar-project-457815/assets/seasonal_baseline_mean

Processing: projects/bar-project-457815/assets/seasonal_baseline_median
  Found 0 images
  Deleted collection: projects/bar-project-457815/assets/seasonal_baseline_median

Processing: projects/bar-project-457815/assets/seasonal_baseline_std
  Found 12 images
  Deleted collection: projects/bar-project-457815/assets/seasonal_baseline_std

Processing: projects/bar-project-457815/assets/monthly_anomaly
  Found 106 images
  Deleted collection: projects/bar-project-457815/assets/monthly_an

In [ ]:
# ============================================================
# Pre-flight check
# Lists target asset paths and reports whether each one
# already exists in the project.
# ============================================================

paths_to_check = [
    ASSET_COLLECTION,
    LABEL_COLLECTION,
    BASELINE_MEAN_FOLDER,
    BASELINE_MEDIAN_FOLDER,
    BASELINE_STD_FOLDER,
    ANOMALY_FOLDER,
]

print(f"Checking target paths for VERSION='{VERSION}':\n")

collisions = []
for path in paths_to_check:
    try:
        info = ee.data.getAsset(path)
        n_imgs = "?"
        try:
            n_imgs = ee.ImageCollection(path).size().getInfo()
        except Exception:
            pass
        print(f"  EXISTS  {path}  (type={info['type']}, n_images={n_imgs})")
        collisions.append(path)
    except Exception:
        print(f"  FREE    {path}")

print()
if collisions:
    print(f"{len(collisions)} target path(s) already exist.")
else:
    print(f"All target paths are free.")

Checking target paths for VERSION='_10m':

  EXISTS  projects/bar-project-457815/assets/s2_monthly_image_collection_10m  (type=IMAGE_COLLECTION, n_images=106)
  EXISTS  projects/bar-project-457815/assets/fire_monthly_labels_10m  (type=IMAGE_COLLECTION, n_images=0)
  EXISTS  projects/bar-project-457815/assets/seasonal_baseline_mean_10m  (type=IMAGE_COLLECTION, n_images=0)
  EXISTS  projects/bar-project-457815/assets/seasonal_baseline_median_10m  (type=IMAGE_COLLECTION, n_images=0)
  EXISTS  projects/bar-project-457815/assets/seasonal_baseline_std_10m  (type=IMAGE_COLLECTION, n_images=0)
  EXISTS  projects/bar-project-457815/assets/monthly_anomaly_10m  (type=IMAGE_COLLECTION, n_images=0)

6 target path(s) already exist.


In [ ]:
# ============================================================
# Step 1 — Load the region of interest, spatial masks, and
# the fire ground truth polygons.
#
# - landuse_mask:     excludes pixels with FCODE == 501
# - human_infra_mask: excludes pixels covered by built infrastructure
# - final_mask:       intersection of both, applied to all imagery
# - fires_clean:      fire polygons within the analysis date range
# ============================================================

working_area = ee.FeatureCollection(ROI_FC_ID)
region       = working_area.geometry()

landuse        = ee.FeatureCollection(LANDUSE_FC_ID)
human_infra_fc = ee.FeatureCollection(HUMAN_INFRA_FC_ID)

landuse_mask = (landuse
                .filter(ee.Filter.neq("FCODE", 501))
                .reduceToImage(["FCODE"], ee.Reducer.first())
                .gt(0))

human_infra_mask = (human_infra_fc
                    .reduceToImage(["Shape_Area"], ee.Reducer.count())
                    .gt(0))

final_mask = landuse_mask.And(human_infra_mask.Not())

fires_truth = (ee.FeatureCollection(FIRES_TRUTH_FC_ID)
               .filterBounds(region)
               .map(lambda f: f.set("geom_type", f.geometry().type()))
               .filter(ee.Filter.inList("geom_type", ["Polygon", "MultiPolygon"]))
               .map(lambda f: f.set({
                    "fire_date":         ee.Date(ee.Number(f.get("FireDate"))),
                    "fire_year":         ee.Date(ee.Number(f.get("FireDate"))).get("year"),
                    "system:time_start": ee.Number(f.get("FireDate"))
               })))

START = ee.Date(DATE_START).millis()
END   = ee.Date(DATE_END).millis()

fires_clean = fires_truth.filter(ee.Filter.rangeContains("FireDate", START, END))

print("Fire polygons within date range:", fires_clean.size().getInfo())

# Binary raster: 1 if pixel falls inside any fire polygon across the full period
burned_any = ee.Image(0).toByte().paint(fires_clean, 1).rename("burned_any")

Fire polygons within date range: 2439


In [ ]:
#Run this cell one time only, to build the collecion:

# Step 2 — Build a clean Sentinel-2 collection.
#
# Bands include the 10 m visible/NIR bands and the 20 m
# red-edge and SWIR bands. Cloud Score+ (cs_cdf) is used
# to mask cloudy pixels at a threshold of CLEAR_THRESH.
# The spatial mask from Step 1 is applied to every image.
# ============================================================
BANDS = ["B4", "B8", "B8A", "B11", "B12"]   # Raw bands needed for downstream indices

s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(region)
      .filterDate(DATE_START, DATE_END)
      .select(BANDS))

csplus = (ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
          .filterBounds(region)
          .filterDate(DATE_START, DATE_END))

# Link cs_cdf score from Cloud Score+ onto each Sentinel-2 image
s2_with_cs = s2.linkCollection(csplus, ["cs_cdf"])

def mask_with_cs(img):
    return (img
            .updateMask(img.select("cs_cdf").gte(CLEAR_THRESH))
            .select(BANDS)
            .updateMask(final_mask)
            .toFloat())

s2_clean = ee.ImageCollection(s2_with_cs.map(mask_with_cs))

print("Clean Sentinel-2 images over ROI:", s2_clean.size().getInfo())

# Verify the native projection of a SWIR band
img  = s2_clean.first()
proj = img.select("B11").projection()
print("CRS:      ", proj.crs().getInfo())
print("Transform:", proj.transform().getInfo())

Clean Sentinel-2 images over ROI: 1307
CRS:       EPSG:32636
Transform: PARAM_MT["Affine", 
  PARAMETER["num_row", 3], 
  PARAMETER["num_col", 3], 
  PARAMETER["elt_0_0", 20.0], 
  PARAMETER["elt_0_2", 600000.0], 
  PARAMETER["elt_1_1", -20.0], 
  PARAMETER["elt_1_2", 3500040.0]]


In [ ]:
# run this cell one time only!
#Step 3 — Build monthly mean composites with seven indices
# and one quality band, exported as separate image assets.
#
# Raw bands retained:
#   B4  (Red)    - needed for NDVI, EVI, BSI
#   B8  (NIR)    - needed for NDVI, NDMI, NBR, EVI
#   B11 (SWIR1)  - needed for NDMI, MIRBI, BSI
#   B12 (SWIR2)  - needed for NBR, MIRBI, BAIS2
#
# Indices included:
#   NDVI   - vegetation greenness
#   EVI    - atmospherically corrected vegetation
#   NDMI   - canopy moisture stress
#   NBR    - normalized burn ratio
#   MIRBI  - mid-infrared burn index
#   BSI    - bare soil index
#   BAIS2  - Sentinel-2 burned area index
#
# Quality band:
#   pct_valid - fraction of cloud-free observations in the month



# Raw bands needed for index computation and as direct features
BANDS = ["B4", "B8", "B8A", "B11", "B12"]

# Indices computed from the raw bands
INDICES = [
    "NDVI",
    "EVI",
    "NDMI",
    "NBR",
    "MIRBI",
    "BSI",
    "BAIS2"
]

def add_indices(img_ref):
    eps = ee.Image.constant(1e-6)

    b4  = img_ref.select("B4")
    b8  = img_ref.select("B8")
    b8a = img_ref.select("B8A")
    b11 = img_ref.select("B11")
    b12 = img_ref.select("B12")

    # Core normalized indices
    ndvi = img_ref.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndmi = img_ref.normalizedDifference(["B8", "B11"]).rename("NDMI")
    nbr  = img_ref.normalizedDifference(["B8", "B12"]).rename("NBR")

    # Enhanced Vegetation Index
    # EVI uses Blue band internally; here we use the standard 2-band formulation
    # that drops the Blue term, suitable when atmospheric correction is reliable
    evi = (b8.subtract(b4).multiply(2.5)
             .divide(b8.add(b4.multiply(2.4)).add(1).add(eps))
             .rename("EVI"))

    # Mid-Infrared Burn Index (Trigg & Flasse 2001)
    mirbi = (b12.multiply(10).subtract(b11.multiply(9.8)).add(2)).rename("MIRBI")

    # Bare Soil Index — uses B11, B4, B8, B2; substituting B8A for B2 here
    # to avoid keeping Blue band; equivalent under typical conditions
    bsi = ((b11.add(b4).subtract(b8.add(b8a)))
           .divide(b11.add(b4).add(b8).add(b8a).add(eps))
           .rename("BSI"))

    # Burned Area Index for Sentinel-2 (Filipponi 2018)
    term1 = ee.Image(1).subtract(b11.multiply(b12).multiply(b8a).divide(b4.add(eps)).sqrt())
    term2 = b12.subtract(b8a).divide(b12.add(b8a).add(eps).sqrt()).add(1)
    bais2 = term1.multiply(term2).rename("BAIS2")

    return img_ref.addBands([ndvi, evi, ndmi, nbr, mirbi, bsi, bais2])

def monthly_mean_image(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end   = start.advance(1, "month")

    ic = s2_clean.filterDate(start, end)

    # Scale Sentinel-2 SR to surface reflectance in [0, 1] and append indices
    ic_ref = ic.map(lambda img:
                    img.select(BANDS).multiply(0.0001)
                       .copyProperties(img, img.propertyNames()))
    ic_ref = ic_ref.map(add_indices)

    comp = ic_ref.select(BANDS + INDICES).mean()

    # Fraction of valid (cloud-free) observations in the month
    n_obs_valid       = ic_ref.select("B4").count()
    n_images_in_month = ee.Number(ic_ref.size())
    pct_valid         = n_obs_valid.divide(n_images_in_month).rename("pct_valid")

    out = (comp
           .addBands(pct_valid)
           .set({
               "year":              year,
               "month":             month,
               "month_start":       start.format("YYYY-MM-dd"),
               "n_images_in_month": n_images_in_month,
               "system:time_start": start.millis()
           })
           .clip(region))
    return out

def export_month_to_asset(year, month):
    img      = monthly_mean_image(year, month)
    ym       = f"{int(year)}_{int(month):02d}"
    desc     = f"s2_monthly_mean_{ym}"
    asset_id = f"{ASSET_COLLECTION}/{desc}"

    task = ee.batch.Export.image.toAsset(
        image            = img,
        description      = desc,
        assetId          = asset_id,
        region           = region,
        scale            = SCALE,
        crs              = CRS,
        maxPixels        = MAX_PIXELS,
        pyramidingPolicy = {'default': 'mean'}
    )
    task.start()
    return task

# Submit export tasks for every month of the analysis period 2017, 2026
tasks = []
for y in range(2024, 2026):
    for m in range(1, 13):
        tasks.append(export_month_to_asset(y, m))

print(f"Submitted {len(tasks)} monthly composite export tasks")
print(f"Bands per image: {len(BANDS) + len(INDICES) + 1} ({len(BANDS)} raw + {len(INDICES)} indices + 1 quality)")
print(f"Target collection: {ASSET_COLLECTION}")

Submitted 24 monthly composite export tasks
Bands per image: 13 (5 raw + 7 indices + 1 quality)
Target collection: projects/bar-project-457815/assets/s2_monthly_image_collection_10m


Finish running "phase 1". wait to complete and then run the rest:

In [ ]:

# Step 4 - Build monthly binary fire-label images from the
# FireDate fire-scar polygons and export each month as a
# separate image asset.

# Each pixel is 1 if any fire polygon active in that month
# overlaps it, and 0 otherwise.

# Run only after the Step 3 export tasks have completed!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


def monthly_fire_label_image(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end   = start.advance(1, "month")

    fires_m = fires_clean.filterDate(start, end)

    label = (ee.Image(0).toByte()
             .paint(fires_m, 1)
             .rename("burned_this_month")
             .clip(region)
             .unmask(0)
             .set({
                 "year":              year,
                 "month":             month,
                 "month_start":       start.format("YYYY-MM-dd"),
                 "n_fire_polygons":   fires_m.size(),
                 "system:time_start": start.millis()
             }))
    return label

def export_monthly_fire_label_to_asset(year, month):
    img      = monthly_fire_label_image(year, month)
    ym       = f"{int(year)}_{int(month):02d}"
    desc     = f"fire_label_{ym}"
    asset_id = f"{LABEL_COLLECTION}/{desc}"

    task = ee.batch.Export.image.toAsset(
        image       = img,
        description = desc,
        assetId     = asset_id,
        region      = region,
        scale       = SCALE,
        crs         = CRS,
        maxPixels   = MAX_PIXELS
    )
    task.start()
    return task

tasks = []
for y in range(2017, 2026):
    for m in range(1, 13):
        tasks.append(export_monthly_fire_label_to_asset(y, m))

print(f"Submitted {len(tasks)} fire-label export tasks")
print(f"Target collection: {LABEL_COLLECTION}")

Submitted 108 fire-label export tasks
Target collection: projects/bar-project-457815/assets/fire_monthly_labels_10m


In [ ]:
# ============================================================
# Step 5 — Compute per-calendar-month baseline statistics
# (mean, median, standard deviation) from the 2017-2019
# reference period, using the monthly composite collection
# exported in Step 3.
#
# Run only after the Step 3 export tasks have completed.
# ============================================================

# Same band schema as the monthly composites
ALL_BANDS = [
    "B4", "B8", "B8A", "B11", "B12",
    "NDVI", "EVI", "NDMI", "NBR", "MIRBI", "BSI", "BAIS2",
    "pct_valid"
]

ic_full = ee.ImageCollection(ASSET_COLLECTION)

# Reference period for baseline computation
ic_ref = ic_full.filter(
    ee.Filter.And(
        ee.Filter.gte("year", 2017),
        ee.Filter.lte("year", 2019)
    )
)
print("Images in reference period (2017-2019):", ic_ref.size().getInfo())

baseline_tasks = []

for month in range(1, 13):
    ic_month = ic_ref.filter(ee.Filter.eq("month", month))
    n = ic_month.size().getInfo()
    print(f"  Month {month:02d}: {n} images")

    # Per-band mean across reference years for this calendar month
    baseline_mean = (ic_month.select(ALL_BANDS).mean()
                     .rename(ALL_BANDS)
                     .set({
                         "month":             month,
                         "baseline_type":     "mean",
                         "reference_years":   "2017-2019",
                         "system:time_start": ee.Date.fromYMD(2018, month, 1).millis()
                     })
                     .clip(region))

    # Per-band median across reference years for this calendar month
    baseline_median = (ic_month.select(ALL_BANDS).median()
                       .rename(ALL_BANDS)
                       .set({
                           "month":             month,
                           "baseline_type":     "median",
                           "reference_years":   "2017-2019",
                           "system:time_start": ee.Date.fromYMD(2018, month, 1).millis()
                       })
                       .clip(region))

    # Per-band standard deviation across reference years for this calendar month
    baseline_std = (ic_month.select(ALL_BANDS)
                    .reduce(ee.Reducer.stdDev())
                    .rename([b + "_stdDev" for b in ALL_BANDS])
                    .rename(ALL_BANDS)
                    .set({
                        "month":             month,
                        "baseline_type":     "std",
                        "reference_years":   "2017-2019",
                        "system:time_start": ee.Date.fromYMD(2018, month, 1).millis()
                    })
                    .clip(region))

    mm = f"{month:02d}"

    t1 = ee.batch.Export.image.toAsset(
        image       = baseline_mean,
        description = f"baseline_mean_{mm}",
        assetId     = f"{BASELINE_MEAN_FOLDER}/baseline_mean_{mm}",
        region      = region, scale = SCALE, crs = CRS, maxPixels = MAX_PIXELS
    )
    t1.start()

    t2 = ee.batch.Export.image.toAsset(
        image       = baseline_median,
        description = f"baseline_median_{mm}",
        assetId     = f"{BASELINE_MEDIAN_FOLDER}/baseline_median_{mm}",
        region      = region, scale = SCALE, crs = CRS, maxPixels = MAX_PIXELS
    )
    t2.start()

    t3 = ee.batch.Export.image.toAsset(
        image       = baseline_std,
        description = f"baseline_std_{mm}",
        assetId     = f"{BASELINE_STD_FOLDER}/baseline_std_{mm}",
        region      = region, scale = SCALE, crs = CRS, maxPixels = MAX_PIXELS
    )
    t3.start()

    baseline_tasks.extend([t1, t2, t3])

print(f"\nSubmitted {len(baseline_tasks)} baseline export tasks")
print(f"Bands per image: {len(ALL_BANDS)}")
print(f"  Mean:   {BASELINE_MEAN_FOLDER}")
print(f"  Median: {BASELINE_MEDIAN_FOLDER}")
print(f"  Std:    {BASELINE_STD_FOLDER}")

Images in reference period (2017-2019): 34
  Month 01: 2 images
  Month 02: 3 images
  Month 03: 2 images
  Month 04: 3 images
  Month 05: 3 images
  Month 06: 3 images
  Month 07: 3 images
  Month 08: 3 images
  Month 09: 3 images
  Month 10: 3 images
  Month 11: 3 images
  Month 12: 3 images

Submitted 36 baseline export tasks
Bands per image: 13
  Mean:   projects/bar-project-457815/assets/seasonal_baseline_mean_10m
  Median: projects/bar-project-457815/assets/seasonal_baseline_median_10m
  Std:    projects/bar-project-457815/assets/seasonal_baseline_std_10m


In [ ]:
#There wasn't enough space on gee, so I will calculate it during the run, and not export to gee
'''
# ============================================================
# Step 6 — Compute monthly anomalies relative to the
# seasonal baseline. For each (year, month) image, three
# representations are exported:
#
#   Z-score anomaly = (observed - baseline_mean) / baseline_std
#   Raw anomaly     = observed - baseline_median
#   dNBR            = baseline_median(NBR) - observed(NBR)
#
# Z-score and raw anomaly are computed on seven indices used
# downstream: NDVI, NBR, MIRBI, NDMI, BSI, BAIS2, EVI.
#
# dNBR follows the convention of Key & Benson (2006) adapted
# to a seasonal-baseline reference: a positive value indicates
# vegetation loss relative to the pixel's typical condition
# for that calendar month.
#
# Run only after the Step 5 export tasks have completed.!!!!!!!!!!!!!!!!!!!!!!!!
# ============================================================

ANOMALY_BANDS = ["NDVI", "NBR", "MIRBI", "NDMI", "BSI", "BAIS2", "EVI"]

ic_full   = ee.ImageCollection(ASSET_COLLECTION)
ic_mean   = ee.ImageCollection(BASELINE_MEAN_FOLDER)
ic_median = ee.ImageCollection(BASELINE_MEDIAN_FOLDER)
ic_std    = ee.ImageCollection(BASELINE_STD_FOLDER)

def get_baseline(ic, month, bands):
    return ee.Image(
        ic.filter(ee.Filter.eq("month", month)).first()
    ).select(bands)

# All (year, month) pairs in the analysis period
MONTHS = []
for y in range(2017, 2026):
    for m in range(1, 13):
        if y == 2017 and m == 1:
            continue
        MONTHS.append((y, m))

anomaly_tasks = []

for (year, month) in MONTHS:
    ym = f"{year}_{month:02d}"

    observed = (ic_full
                .filter(ee.Filter.eq("year",  year))
                .filter(ee.Filter.eq("month", month))
                .first())

    observed = ee.Image(observed).select(ANOMALY_BANDS).unmask(-9999)

    b_mean   = get_baseline(ic_mean,   month, ANOMALY_BANDS)
    b_median = get_baseline(ic_median, month, ANOMALY_BANDS)
    b_std    = get_baseline(ic_std,    month, ANOMALY_BANDS)

    eps = ee.Image.constant(1e-6)

    z_score = (observed
               .subtract(b_mean)
               .divide(b_std.add(eps))
               .rename([b + "_zscore" for b in ANOMALY_BANDS]))

    raw_anomaly = (observed
                   .subtract(b_median)
                   .rename([b + "_anomaly" for b in ANOMALY_BANDS]))

    # dNBR relative to the seasonal baseline median
    # Sign convention: positive dNBR = vegetation loss
    dnbr = (b_median.select("NBR")
            .subtract(observed.select("NBR"))
            .rename("NBR_dnbr"))

    # Final image: 7 z-score + 7 raw anomaly + 1 dNBR = 15 bands
    anomaly_img = (z_score
                   .addBands(raw_anomaly)
                   .addBands(dnbr)
                   .set({
                       "year":              year,
                       "month":             month,
                       "system:time_start": ee.Date.fromYMD(year, month, 1).millis()
                   })
                   .clip(region))

    task = ee.batch.Export.image.toAsset(
        image       = anomaly_img,
        description = f"anomaly_{ym}",
        assetId     = f"{ANOMALY_FOLDER}/anomaly_{ym}",
        region      = region,
        scale       = SCALE,
        crs         = CRS,
        maxPixels   = MAX_PIXELS
    )
    task.start()
    anomaly_tasks.append(task)

print(f"Submitted {len(anomaly_tasks)} anomaly export tasks")
print(f"Target collection: {ANOMALY_FOLDER}")
print(f"Bands per image:   {len(ANOMALY_BANDS) * 2 + 1} (7 z-score + 7 raw anomaly + 1 dNBR)")
'''

Submitted 107 anomaly export tasks
Target collection: projects/bar-project-457815/assets/monthly_anomaly_10m
Bands per image:   15 (7 z-score + 7 raw anomaly + 1 dNBR)



## Changed Pixel Resolution to 10m

I have updated the `SCALE` variable in all relevant code cells from 20m to 10m. This change will ensure that all subsequent image processing, exports, and analyses are performed at a 10-meter pixel resolution, providing finer detail in the results.